# Preparação dos dados: Travel Agency Data

Fonte: [Kaggle — nhatnguyentran/travel-agency-data](https://www.kaggle.com/datasets/nhatnguyentran/travel-agency-data).

Buscas de hotel feitas por clientes de uma agência de viagens entre jan/2013 e dez/2014, indicando se a busca virou reserva (`is_booking`). 100 mil linhas, 25 colunas.

**Como usar:** baixe o CSV no Kaggle (botão *Download*), descompacte e coloque o `.csv` em `data/raw/`. Depois rode todas as células. A última exporta `data/clean/dataset.csv`, que é o arquivo lido pelo `app.py`.

O tratamento em si está em `src/preparacao.py` (função `preparar`), para ficar igual entre o notebook e o app.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

# Raiz do projeto (o notebook roda de dentro de notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

from src.preparacao import COLUNAS_SENSIVEIS, preparar

PASTA_RAW = RAIZ / "data" / "raw"
SAIDA = RAIZ / "data" / "clean" / "dataset.csv"

## Carregando o dataset bruto

In [ ]:
arquivos = sorted(PASTA_RAW.glob("*.csv")) or sorted(PASTA_RAW.glob("*.zip"))
assert arquivos, f"Coloque o CSV baixado do Kaggle em {PASTA_RAW}"
print("Lendo:", arquivos[0].name)

raw = pd.read_csv(arquivos[0])
raw.head()

## Princípio estrutural

In [ ]:
raw.shape

In [ ]:
raw.info()

In [ ]:
raw.isna().sum().sort_values(ascending=False).head(10)

`orig_destination_distance` é a única coluna com muitos ausentes: a distância só é registrada quando a agência consegue localizar o usuário. Os ausentes **não** são imputados; o app ignora essas linhas só nos gráficos dessa variável.

In [ ]:
print("Duplicadas:", raw.duplicated().sum())

## Tratamento

O que `preparar()` faz:

1. Remove a coluna de índice `Unnamed: 0` e as linhas duplicadas.
2. **Privacidade:** remove `user_id`, `user_location_region` e `user_location_city` (identificam o usuário ou sua localização fina e não são usadas no app).
3. Converte `date_time`, `srch_ci` e `srch_co` para data; descarta datas inválidas.
4. Cria variáveis derivadas: `antecedencia_dias` (busca → check-in), `noites` (check-in → check-out) e `total_hospedes`.
5. Descarta buscas impossíveis: check-in antes da busca, 0 noites, nenhum hóspede ou nenhum quarto.
6. Cria rótulos legíveis: `resultado`, `dispositivo`, `tipo_compra`, `canal`, `mes`, `faixa_antecedencia`.

In [ ]:
df = preparar(raw)
print(f"{len(raw):,} linhas brutas → {len(df):,} linhas tratadas ({len(raw) - len(df):,} removidas)")
print("Colunas sensíveis removidas:", [c for c in COLUNAS_SENSIVEIS if c in raw.columns])

## Princípio distribucional (variáveis quantitativas)

In [ ]:
VARS_QUANT = [
    "orig_destination_distance", "antecedencia_dias", "noites",
    "srch_adults_cnt", "srch_children_cnt", "srch_rm_cnt", "cnt",
]

desc = df[VARS_QUANT].describe().T
desc["skewness"] = df[VARS_QUANT].skew()
desc.round(2)

In [ ]:
print("Taxa de conversão geral:", f"{df['is_booking'].mean():.1%}")
df["resultado"].value_counts()

## Exportando o dado tratado

In [ ]:
SAIDA.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(SAIDA, index=False)
print(f"Salvo em {SAIDA.relative_to(RAIZ)} ({SAIDA.stat().st_size / 1e6:.1f} MB)")